# 15 Files

| | |
|---|---|
| **Path** | Python Fundamentals |
| **Prerequisite** | 14 |

A file is named data on disk. `pathlib.Path` is the modern way to name it. `write_text` and `read_text` move a whole string in one call. `with` opens a file and closes it when the block ends, including when the block raises.

The examples use a temporary directory, so running the lesson does not leave files behind. The calls are the same calls you will use on a real path.


## Example 1 — Write a string, then read it back

### Predict

Two lines are written, including their newline characters. `splitlines` returns the two names and drops the newlines.

### Run


In [1]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "notes.txt"
    # write_text stores the string. read_text brings it back.
    path.write_text("Ada\nGrace\n", encoding="utf-8")
    text = path.read_text(encoding="utf-8")
    print(text.splitlines())
    print(path.name)


['Ada', 'Grace']
notes.txt


## Example 2 — with closes the file, then you can still read the path

### Predict

Inside the block, one more name is appended. After the block, `handle.closed` is true. The path can still be read, because closing the handle is not deleting the file.

### Run


In [2]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "notes.txt"
    path.write_text("Ada\n", encoding="utf-8")
    # "a" appends. Leaving the with block closes the handle.
    with path.open("a", encoding="utf-8") as handle:
        handle.write("Grace\n")
    print(handle.closed)
    print(path.read_text(encoding="utf-8").splitlines())


True
['Ada', 'Grace']


## Example 3 — A tiny table, written as text and read back

### Predict

Each row is a line. Commas separate the fields. Splitting each line on the comma recovers the name and the year. The year comes back as text until `int` converts it.

### Run


In [3]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "people.csv"
    path.write_text("Ada,1815\nGrace,1906\n", encoding="utf-8")
    people = []
    for line in path.read_text(encoding="utf-8").splitlines():
        # The year is still text until int converts it.
        name, year = line.split(",")
        people.append((name, int(year)))
    print(people)


[('Ada', 1815), ('Grace', 1906)]


## Example 4 — The csv module handles the splitting

### Predict

`csv.reader` yields rows that are already lists of fields. The header is the first row. The next row starts with `Ada`.

### Run


In [4]:
import csv
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "people.csv"
    path.write_text("name,year\nAda,1815\nGrace,1906\n", encoding="utf-8")
    # newline="" lets the csv module handle line endings itself.
    with path.open(encoding="utf-8", newline="") as handle:
        rows = list(csv.reader(handle))
    print(rows[0])
    print(rows[1])


['name', 'year']
['Ada', '1815']


## A pitfall

A path that does not exist cannot be read. The error is `FileNotFoundError`. Creating the parent directory, or checking `path.exists()`, belongs in the program before the read. This cell prints the error instead of crashing the lesson.

### Run


In [5]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as folder:
    missing = Path(folder) / "missing.txt"
    print(missing.exists())
    try:
        missing.read_text(encoding="utf-8")
    except FileNotFoundError as error:
        print(type(error).__name__)
        # The message names the file that was requested.
        print(error.filename.endswith("missing.txt"))


False
FileNotFoundError
True


## Summary

Name a file with `Path`, and pass `encoding="utf-8"` when you read or write text. `with` closes the handle even if the block fails. Split simple lines yourself when the format is truly simple, and use `csv` when the file is a table. A missing file raises `FileNotFoundError`.
